#### Reading posthoc CSV stats
Start 5/5/25. To gather all posthoc csvs exported, unite into one file
v2-5/31/26- add supp figure handling and addition into megafolder 

#### Setup


In [35]:
%pip install -r requirements.txt
from pathlib import Path
import os
import notebook_setup
info = notebook_setup.setup()

# Environment & Imports Setup
import matplotlib as matplotlib
%matplotlib inline 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from datetime import datetime
import itertools
import ast
##customs 
from helper_functions import *

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


## Preprocess/Gather Files

#### Read and gather all datasets 

In [36]:
#Set/create save and load folder paths 
here = info["repo_root"]
print(f" Here: {here}")
results_location = Path(here).parents[0] / "results"
data_location = Path(here).parents[0] / "data"
print(f"Saving results in {results_location}. Data location is {data_location}")
csv_folder_most_recent = results_location/ f"analysis_CSV_output/" #folders that analysis output goes to
os.chdir(results_location)
print(os.getcwd())


 Here: c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\code
Saving results in c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\results. Data location is c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\data
c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\results


In [37]:
folders = []
with os.scandir(os.getcwd()) as dir_iter:
    for item in dir_iter:
        # print(item)
        if item.is_dir():
            folders.append(item)
folders

[<DirEntry 'analysis_CSV_output'>,
 <DirEntry 'CCG_compare_12_Feb_2026 w old_01_21_2026_new_01_31_2026'>,
 <DirEntry 'CCG_compare_12_Feb_2026 w old_01_21_2026_new_02_12_2026'>,
 <DirEntry 'CCG_compare_16_Feb_2026 w old_01_21_2026_new_02_16_2026'>,
 <DirEntry 'CCG_compare_24_Feb_2026 w old_01_21_2026_new_02_16_2026'>,
 <DirEntry 'CCG_compare_25_Feb_2026 w old_01_21_2026_new_02_16_2026'>,
 <DirEntry 'CCG_compare_26_Jan_2026 w old_01_21_2026_new_01_24_2026'>,
 <DirEntry 'CCG_compare_26_Jan_2026 w old_05_31_2025_new_01_23_2026'>,
 <DirEntry 'CCG_compare_26_Jan_2026 w old_05_31_2025_new_01_24_2026'>,
 <DirEntry 'CCG_compare_29_Jan_2026 w old_01_21_2026_new_01_24_2026'>,
 <DirEntry 'CCG_compare_31_Jan_2026 w old_01_21_2026_new_01_31_2026'>,
 <DirEntry 'CCG_comparison_25_Jan_2026'>,
 <DirEntry 'CCG_comparison_26_Jan_2026'>,
 <DirEntry 'date_sorted_figures'>,
 <DirEntry 'dff_baseline_zscore_norm_ensemble_detection_01-Dec-2025_5000 shuffles'>,
 <DirEntry 'dff_baseline_zscore_norm_ensemble_detec

#### Sort folders in directory 

In [38]:
#sort folders in dir by last edited
sorted_folders = sorted(folders, reverse = True, key = lambda entry: entry.stat().st_mtime)
sorted_folders[:10]

[<DirEntry 'analysis_CSV_output'>,
 <DirEntry 'supp_fig_4'>,
 <DirEntry 'revision_fig_1'>,
 <DirEntry 'fig_4'>,
 <DirEntry 'fig_3'>,
 <DirEntry 'supp_fig_2'>,
 <DirEntry 'fig_7'>,
 <DirEntry 'fig_6'>,
 <DirEntry 'date_sorted_figures'>,
 <DirEntry 'fig_5'>]

In [39]:
sorted_csv_storage = [c for c in sorted_folders if 'analysis_CSV_output' in c.name]
print(sorted_csv_storage)
most_recent_csv_storage = sorted_csv_storage[0]
most_recent_csv_storage

[<DirEntry 'analysis_CSV_output'>]


<DirEntry 'analysis_CSV_output'>

#### get CSV folders in dir

In [40]:
## navigate
os.chdir(most_recent_csv_storage)
folder = os.getcwd()
print(folder)
csv_files = [f for f in os.listdir(folder) if f.lower().endswith(".csv")]
print(f" {len(csv_files)} csv files found in folder")

c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\results\analysis_CSV_output
 798 csv files found in folder


In [41]:
csv_files

['1_# Perseverative Errors_behav_posthoc MWU_06_Apr_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_06_May_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_16_Mar_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_18_Mar_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_19_Mar_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_24_Feb_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_25_Feb_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_27_Feb_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_28_Feb_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_28_Mar_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_29_Mar_2026.csv',
 '1_behavior_perseverative_error_anova_06_Apr_2026.csv',
 '1_behavior_perseverative_error_anova_06_May_2026.csv',
 '1_behavior_perseverative_error_anova_16_Mar_2026.csv',
 '1_behavior_perseverative_error_anova_18_Mar_2026.csv',
 '1_behavior_perseverative_error_anova_19_Mar_2026.csv',
 '1_behavior_persevera

#### find latest run of data 

In [42]:
def nearest(items, pivot):
    return min(items, key=lambda x: abs(x - pivot)) 
    
def extract_date_str(csv_str, suffix = '.csv', split_delim = '_', get_post_split = -3):
    return csv_str.split(suffix)[0].split(split_delim)[get_post_split:]

In [43]:
date_list = [" ".join(extract_date_str(x)) for x in csv_files ] #drop .csv, then get last 3 entries, but only if has any digit in str
dates_clean = [x for x in date_list if sum(i.isdigit() for i in x) > 5]
dates_clean[-10:]

['19 Mar 2026',
 '23 Feb 2026',
 '25 Feb 2026',
 '26 May 2026',
 '28 Mar 2026',
 '11 May 2026',
 '11 May 2026',
 '04 Jun 2026',
 '11 May 2026',
 '26 May 2026']

In [44]:
now = datetime.today()
print(now)

2026-06-08 14:10:17.518780


#### for each unique fig number, find the csvs with the min timedelta datetime, and collect

In [45]:
def store_csv_name_by_fig_num(csv_files, max_fig_num = 8):
    ''' To- create dict where key = fig num and val = list of csv names with dates in title'''
    num_store = {f: list() for f in range(max_fig_num)}
    ## ## loop throuhg all CSVs
    for f in csv_files:
        fig_n = f.split("_")[0]
        if all((i.isdigit() for i in fig_n)):        # print(fig_n)
            num_store[int(fig_n)].append(f)
    return num_store

In [46]:
num_store = store_csv_name_by_fig_num(csv_files)
num_store

{0: [],
 1: ['1_# Perseverative Errors_behav_posthoc MWU_06_Apr_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_06_May_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_16_Mar_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_18_Mar_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_19_Mar_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_24_Feb_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_25_Feb_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_27_Feb_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_28_Feb_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_28_Mar_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_29_Mar_2026.csv',
  '1_behavior_perseverative_error_anova_06_Apr_2026.csv',
  '1_behavior_perseverative_error_anova_06_May_2026.csv',
  '1_behavior_perseverative_error_anova_16_Mar_2026.csv',
  '1_behavior_perseverative_error_anova_18_Mar_2026.csv',
  '1_behavior_perseverative_error_anova_19_Mar_2026.c

#### create csv store


In [47]:
results_location

WindowsPath('c:/Users/13car/Dropbox/local_github_repos_personal/dlx56_mPFC_1p_SohalLab/results')

In [48]:
csv_store_folder = results_location / "figure_tables"
make_folder(csv_store_folder)

The folder 'c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\results\figure_tables' already exists.


## Combine/Save Tables

#### test combination using figure 3 concat:

In [49]:
#
def get_last_fig_csv_names(num_store:dict, fig_num:int, skip_flag = ['anova', 'cells active per trial'],**kwargs):
    fig_storage = num_store[fig_num]
    ##processing logic 
    closest_time, last_datetime_str = get_latest_csv_datetime(fig_storage,skip_flag = skip_flag)##get list of dates in csvs for figure of interest, then find closest
    current_files = get_fig_csv_matching_datetime(fig_storage, closest_time,skip_flag = skip_flag)
    return current_files, closest_time
    
#get list of dates in csvs for figure of interest, then find closest
def get_latest_csv_datetime(fig_storage:list,skip_flag:list = ["skip"], **kwargs):
    ''' To iterate over list of .csv filenames (with datetime tags embedderd) and find the closest embedded tag to the current time. 
    Skip_flag: list of strings to iterate through and not include in csv list if present'''
    now = datetime.today()
    valid_figs = [x for x in fig_storage if all([skip.lower() not in x.lower() for skip in skip_flag])] #skip_flag: list of str, verify all not present

    date_list_datetime = [datetime.strptime(" ".join(extract_date_str(x)),'%d %b %Y') for x in valid_figs]
    closest_time = min(date_list_datetime , key = lambda x: now- x )
    last_datetime_str = closest_time.strftime('%d_%b_%Y')#convert to str to match save formatting
    print(f"closest datetime: {closest_time}. last_datetime_str = {last_datetime_str}")
    return closest_time, last_datetime_str

def get_fig_csv_matching_datetime(fig_storage, closest_time=datetime.today(), skip_flag:list = ["skip"],):
    ''' To loop over input list of .csv filenames, and pick entries matching the previously found closest datetime'''
    current_files = list()
    #optional- filter list and ignore ones with certain content
    valid_figs = [x for x in fig_storage if all([skip.lower() not in x.lower() for skip in skip_flag])] #skip_flag: list of str, verify all not present
    # print(valid_figs)
    for f in valid_figs:## given closest datetime, iterate and extract 
        f_date = datetime.strptime(" ".join(extract_date_str(f)),'%d %b %Y')# #get string date and convert to datetime to compare against last known date entry
        if f_date == closest_time:
            print(f'matching date file: {f}')
            current_files.append(f)
    
    return current_files


In [50]:
def concat_clean_csv_df(current_files, **kwargs):
    current_dfs = []
    for f in current_files:
        df = pd.read_csv(f)
        df['csv_date'] = " ".join(extract_date_str(os.path.basename(f)))  # e.g. '26 May 2026'
        df['csv_filename'] = os.path.basename(f)  # optional, for full traceability
        current_dfs.append(df)
    combined_fig_df = pd.concat(current_dfs)

    cols_to_drop = ['g_1_child_x', 'g_1_child_y', 'y_is_nonnan', 'y_is_2_elem', 'hue_is_x_axis',
                    'group_1_order_pos', 'group_2_order_pos', 'g_2_child_x', 'g_2_child_y',
                    'tick_text', 'tick_pos', 'hue_group_1_locs', 'hue_group_2_locs', 'Unnamed: 0',
                    'g1_num_loc', 'g2_num_loc', 'g1_cat_loc', 'g2_cat_loc', 'max_group_loc_val', 'plot_name']
    combined_fig_df.drop([c for c in cols_to_drop if c in combined_fig_df.columns], axis=1, inplace=True)
    return combined_fig_df


In [51]:
get_latest_csv_datetime(num_store[1],skip_flag = ['anova', 'trial'])

closest datetime: 2026-05-25 00:00:00. last_datetime_str = 25_May_2026


(datetime.datetime(2026, 5, 25, 0, 0), '25_May_2026')

In [52]:
fig_num = 3
current_files, closest_time = get_last_fig_csv_names(num_store, fig_num)
combined_fig_df= concat_clean_csv_df(current_files)

combined_fig_df.group_1_n = combined_fig_df.group_1_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)
combined_fig_df.group_2_n = combined_fig_df.group_2_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)

#key value store for old: new col name
clean_col_name_dict = {'category_compared_within': "Group of posthoc comparison", 
                       'group_1': "Group 1",
                       'group_2': "Group 2", 
                       'group_1_n': "Group 1 N",
                       'group_2_n': "Group 2 N", 
                       'group_1_mean': "Group 1 Mean",
                       'group_1_sem': "Group 1 SEM",
                       'group_2_mean': "Group 2 Mean",
                       'group_2_sem':"Group 2 SEM",
                       'test_name': "Name of Statistical Test",
                       'stat_result': "Test Result",
                       'pvalue': "Test p-value",
                       'categorical_subgroup': "Alternate name- group compared within",
                       'numeric_var': "Variable Compared between Groups",
                       'hue_var': "Variable labeling post-hoc group",
                       'x_category_var': "Categorical variable of plot (x-axis)",
                       'date_tag': "Date of figure creation",
                       'fig_name': "filename of source table",
                       'fig_num': "Figure number"
                      }

combined_fig_df.rename(clean_col_name_dict, axis = 1,inplace = True)
combined_fig_df

closest datetime: 2026-06-04 00:00:00. last_datetime_str = 04_Jun_2026
matching date file: 3_null_CCG uni-class pred- Train IA test RS - Early_RS_Correct ensemble_predict_posthoc cohen d_04_Jun_2026.csv
matching date file: 3_null_CCGP- Train on IA trials, Test on RS trials_Early stage ens_ _ccgp_all_posthoc cohen d_04_Jun_2026.csv


,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,Date of figure creation,filename of source table,Figure number,csv_date,csv_filename,group_1_null_low,group_1_null_high,group_2_null_low,group_2_null_high,exceeds_null
0,Early RS Error,WT VEH,Het VEH,10000,10000,94.3869,2.4220,20.0621,3.8066,robust_cohen_d,...,04_Jun_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,04 Jun 2026,3_null_CCG uni-class pred- Train IA test RS - ...,NaN,NaN,NaN,NaN,NaN
1,Early RS Error,Het VEH,Het CLNZ,10000,10000,20.0621,3.8066,46.2551,2.0052,robust_cohen_d,...,04_Jun_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,04 Jun 2026,3_null_CCG uni-class pred- Train IA test RS - ...,NaN,NaN,NaN,NaN,NaN
2,Early RS Error,Het VEH,Het postCLNZ,10000,10000,20.0621,3.8066,62.2139,2.7931,robust_cohen_d,...,04_Jun_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,04 Jun 2026,3_null_CCG uni-class pred- Train IA test RS - ...,NaN,NaN,NaN,NaN,NaN
3,Early RS Correct,WT VEH,Het VEH,10000,10000,54.4617,2.1427,53.7853,2.4082,robust_cohen_d,...,04_Jun_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,04 Jun 2026,3_null_CCG uni-class pred- Train IA test RS - ...,NaN,NaN,NaN,NaN,NaN
4,Early RS Correct,Het VEH,Het CLNZ,10000,10000,53.7853,2.4082,65.6328,2.4321,robust_cohen_d,...,04_Jun_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,04 Jun 2026,3_null_CCG uni-class pred- Train IA test RS - ...,NaN,NaN,NaN,NaN,NaN
5,Early RS Correct,Het VEH,Het postCLNZ,10000,10000,53.7853,2.4082,75.1442,2.2828,robust_cohen_d,...,04_Jun_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,04 Jun 2026,3_null_CCG uni-class pred- Train IA test RS - ...,NaN,NaN,NaN,NaN,NaN
0,Early_IA_Correct,WT VEH,Het VEH,10000,10000,0.4457,0.0072,0.4566,0.0240,robust_cohen_d,...,04_Jun_2026,"null_CCGP- Train on IA trials, Test on RS tria...",3,04 Jun 2026,"3_null_CCGP- Train on IA trials, Test on RS tr...",0.467529,0.532227,0.422363,0.577148,True
1,Early_IA_Correct,Het VEH,Het CLNZ,10000,10000,0.4566,0.0240,0.5675,0.0179,robust_cohen_d,...,04_Jun_2026,"null_CCGP- Train on IA trials, Test on RS tria...",3,04 Jun 2026,"3_null_CCGP- Train on IA trials, Test on RS tr...",0.422363,0.577148,0.444092,0.557129,True
2,Early_IA_Correct,Het VEH,Het postCLNZ,10000,10000,0.4566,0.0240,0.5456,0.0142,robust_cohen_d,...,04_Jun_2026,"null_CCGP- Train on IA trials, Test on RS tria...",3,04 Jun 2026,"3_null_CCGP- Train on IA trials, Test on RS tr...",0.422363,0.577148,0.433105,0.567871,False
3,Early_IA_Error,WT VEH,Het VEH,10000,10000,0.4873,0.0137,0.4175,0.0099,robust_cohen_d,...,04_Jun_2026,"null_CCGP- Train on IA trials, Test on RS tria...",3,04 Jun 2026,"3_null_CCGP- Train on IA trials, Test on RS tr...",0.435059,0.567383,0.451660,0.548828,True


In [53]:
csv_name = f"fig_{fig_num}_results.csv"
combined_fig_df.to_csv( csv_store_folder/csv_name)

#### Concat posthoc statistics from figures 3-7:

In [54]:
## as figs 1 and 2 don't use traditional posthoc test df outputting, skip that for now 
fig_start = 2
fig_end = 8
##
figs_to_concat = [k for k, v in num_store.items() if v]
print(f" Combining csvs with keys: {figs_to_concat}")
all_fig_tables = []
for fig_num in figs_to_concat:
    current_files, closest_time = get_last_fig_csv_names(num_store, fig_num)
    combined_fig_df= concat_clean_csv_df(current_files)
    combined_fig_df.group_1_n = combined_fig_df.group_1_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)
    combined_fig_df.group_2_n = combined_fig_df.group_2_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)
    #key value store for old: new col name
    clean_col_name_dict = {'category_compared_within': "Group of posthoc comparison", 
                           'group_1': "Group 1",
                           'group_2': "Group 2", 
                           'group_1_n': "Group 1 N",
                           'group_2_n': "Group 2 N", 
                           'group_1_mean': "Group 1 Mean",
                           'group_1_sem': "Group 1 SEM",
                           'group_2_mean': "Group 2 Mean",
                           'group_2_sem':"Group 2 SEM",
                           'test_name': "Name of Statistical Test",
                           'stat_result': "Test Result",
                           'pvalue': "Test p-value",
                           'categorical_subgroup': "Alternate name- group compared within",
                           'numeric_var': "Variable Compared between Groups",
                           'hue_var': "Variable labeling post-hoc group",
                           'x_category_var': "Categorical variable of plot (x-axis)",
                           'date_tag': "Date of figure creation",
                           'fig_name': "filename of source table",
                           'fig_num': "Figure number"
                          }
    combined_fig_df.rename(clean_col_name_dict, axis = 1,inplace = True)
    all_fig_tables.append(combined_fig_df)
    csv_name = f"fig_{fig_num}_concat_results.csv"
    combined_fig_df.to_csv( csv_store_folder/ csv_name)
full_fig_table = pd.concat(all_fig_tables)


 Combining csvs with keys: [1, 2, 3, 4, 5, 6, 7]
closest datetime: 2026-05-25 00:00:00. last_datetime_str = 25_May_2026
matching date file: 1_s_Mean % of cells active in stage that are in stage ensemble_posthoc MWU_25_May_2026.csv
matching date file: 1_s_supp_Proportion frames active per trial_posthoc permutation test_25_May_2026.csv
matching date file: 1_s_supp_wtclnz_pointplot_Mean event rate by phase_posthoc MWU_25_May_2026.csv
closest datetime: 2026-06-04 00:00:00. last_datetime_str = 04_Jun_2026
matching date file: 2_s_supp_CCG + null 1_class pred-Train_Correct_Test_Error- Early_RS_Error ensemble_supp_predict_posthoc cohen d_04_Jun_2026.csv
matching date file: 2_s_supp_CCG + null 1_class pred-Train_Correct_Test_Error-Early_IA_Correct ensemble_supp_predict_posthoc cohen d_04_Jun_2026.csv
matching date file: 2_s_supp_CCG+null single class pred- Train IA test RS - Early_RS_Correct ensemble_supp_predict_posthoc cohen d_04_Jun_2026.csv
closest datetime: 2026-06-04 00:00:00. last_dateti

In [55]:
full_fig_table

,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,Date of figure creation,filename of source table,Figure number,csv_date,csv_filename,group_1_null_low,group_1_null_high,group_2_null_low,group_2_null_high,exceeds_null
0,Early_IA_Correct,WT VEH,WT CLNZ,29,21,58.7353,2.9926,61.6694,3.3390,MWU,...,25_May_2026,Mean % of cells active in stage that are in st...,1_s,25 May 2026,1_s_Mean % of cells active in stage that are i...,NaN,NaN,NaN,NaN,NaN
1,Early_IA_Correct,WT VEH,Het VEH,29,27,58.7353,2.9926,69.5951,2.8289,MWU,...,25_May_2026,Mean % of cells active in stage that are in st...,1_s,25 May 2026,1_s_Mean % of cells active in stage that are i...,NaN,NaN,NaN,NaN,NaN
2,Early_IA_Correct,Het VEH,Het CLNZ,27,21,69.5951,2.8289,69.4418,2.9976,MWU,...,25_May_2026,Mean % of cells active in stage that are in st...,1_s,25 May 2026,1_s_Mean % of cells active in stage that are i...,NaN,NaN,NaN,NaN,NaN
3,Early_IA_Correct,Het VEH,Het postCLNZ,27,26,69.5951,2.8289,70.1219,2.1584,MWU,...,25_May_2026,Mean % of cells active in stage that are in st...,1_s,25 May 2026,1_s_Mean % of cells active in stage that are i...,NaN,NaN,NaN,NaN,NaN
4,Early_IA_Error,WT VEH,WT CLNZ,11,9,76.8790,6.4214,72.8287,6.0399,MWU,...,25_May_2026,Mean % of cells active in stage that are in st...,1_s,25 May 2026,1_s_Mean % of cells active in stage that are i...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,Early_RS_Correct,Het VEH,Het CLNZ,10000,10000,0.9532,0.0062,0.9669,0.0053,robust_cohen_d,...,04_Jun_2026,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,7,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN
2,Early_RS_Correct,Het VEH,Het postCLNZ,10000,10000,0.9532,0.0062,0.9769,0.0044,robust_cohen_d,...,04_Jun_2026,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,7,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN
3,Late_IA,WT VEH,Het VEH,10000,10000,0.9696,0.0051,0.9646,0.0053,robust_cohen_d,...,04_Jun_2026,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,7,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN
4,Late_IA,Het VEH,Het CLNZ,10000,10000,0.9646,0.0053,0.9668,0.0052,robust_cohen_d,...,04_Jun_2026,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,7,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN


In [56]:
## save NON STANDARD tables to separate csv for review, then drop from main table
non_standard = full_fig_table[full_fig_table['filename of source table'].isna()]#.dropna(axis=1, how='all')
non_standard.to_csv(csv_store_folder / "non_standard_figure_tables.csv", index=False)
non_standard


,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,Date of figure creation,filename of source table,Figure number,csv_date,csv_filename,group_1_null_low,group_1_null_high,group_2_null_low,group_2_null_high,exceeds_null


In [57]:
full_fig_table = full_fig_table.dropna(subset=['filename of source table']).dropna(axis=1, how='all')
full_fig_table


,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,Date of figure creation,filename of source table,Figure number,csv_date,csv_filename,group_1_null_low,group_1_null_high,group_2_null_low,group_2_null_high,exceeds_null
0,Early_IA_Correct,WT VEH,WT CLNZ,29,21,58.7353,2.9926,61.6694,3.3390,MWU,...,25_May_2026,Mean % of cells active in stage that are in st...,1_s,25 May 2026,1_s_Mean % of cells active in stage that are i...,NaN,NaN,NaN,NaN,NaN
1,Early_IA_Correct,WT VEH,Het VEH,29,27,58.7353,2.9926,69.5951,2.8289,MWU,...,25_May_2026,Mean % of cells active in stage that are in st...,1_s,25 May 2026,1_s_Mean % of cells active in stage that are i...,NaN,NaN,NaN,NaN,NaN
2,Early_IA_Correct,Het VEH,Het CLNZ,27,21,69.5951,2.8289,69.4418,2.9976,MWU,...,25_May_2026,Mean % of cells active in stage that are in st...,1_s,25 May 2026,1_s_Mean % of cells active in stage that are i...,NaN,NaN,NaN,NaN,NaN
3,Early_IA_Correct,Het VEH,Het postCLNZ,27,26,69.5951,2.8289,70.1219,2.1584,MWU,...,25_May_2026,Mean % of cells active in stage that are in st...,1_s,25 May 2026,1_s_Mean % of cells active in stage that are i...,NaN,NaN,NaN,NaN,NaN
4,Early_IA_Error,WT VEH,WT CLNZ,11,9,76.8790,6.4214,72.8287,6.0399,MWU,...,25_May_2026,Mean % of cells active in stage that are in st...,1_s,25 May 2026,1_s_Mean % of cells active in stage that are i...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,Early_RS_Correct,Het VEH,Het CLNZ,10000,10000,0.9532,0.0062,0.9669,0.0053,robust_cohen_d,...,04_Jun_2026,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,7,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN
2,Early_RS_Correct,Het VEH,Het postCLNZ,10000,10000,0.9532,0.0062,0.9769,0.0044,robust_cohen_d,...,04_Jun_2026,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,7,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN
3,Late_IA,WT VEH,Het VEH,10000,10000,0.9696,0.0051,0.9646,0.0053,robust_cohen_d,...,04_Jun_2026,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,7,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN
4,Late_IA,Het VEH,Het CLNZ,10000,10000,0.9646,0.0053,0.9668,0.0052,robust_cohen_d,...,04_Jun_2026,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,7,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN


In [58]:
unique_figs= sorted(full_fig_table['filename of source table'].unique())
# unique_figs

## create dict mapping key (filename) to value (corresponding figure panel)
map_panel_to_filename = {'Autoencoder Early_IA_Correct_v_Early_RS_Correct DB index by ensembles': "6G" ,
 'Autoencoder Early_IA_Correct_v_Late_IA DB index by ensembles': "7D",
 'Autoencoder Early_IA_Error_v_Early_RS_Error DB index by ensembles': "5H" ,
 'Autoencoder Late_IA_v_Early_RS_Correct DB index by ensembles': "7H" ,
 'CCG single class pred- Train Correct test Error- Early_IA_Correct ensemble': "4C" ,
 'CCG single class pred- Train Correct test Error- Early_RS_Error ensemble': "4F" ,
 'CCG single class pred- Train IA test RS - Early_RS_Correct ensemble': "3C" ,
 'Early_IA_Correct_v_Early_RS_Correct SVM accuracy by ensem': "6C" ,
 'Early_IA_Correct_v_Late_IA SVM accuracy by ensem': "7C" ,
 'Early_IA_Error_v_Early_RS_Error SVM accuracy by ensem': "5C" ,
 'Late_IA_v_Early_RS_Correct SVM accuracy by ensem': "7G" ,
 'time-dep decoding - Early RS Correct ens- Early_IA_Correct_v_Early_RS_Correct': "6D" ,
 'time-dep decoding - Early RS Error ens- Early_IA_Error_v_Early_RS_Error': "5D"
                        }
map_supp_panel_to_file = { 'Mean % of cells active in stage that are in stage ensemble':"S1I",
                           'supp_wtclnz_pointplot_Mean event rate by phase':"S1C",
                           'supp_Proportion frames active per trial': "S1B",
 # supp Autoencoder DB-index panels: land on Supplementary Figure 5
    'supp_Autoencoder Early_IA_Correct_v_Early_RS_Correct DB index by ensembles': "S5I" ,
 'supp_Autoencoder Early_IA_Error_v_Early_RS_Error DB index by ensembles': "S5G" ,
 # supp CCGP uniproportion-CCG prediction panels: land on Supplementary Figure 4
 'supp_CCG single class pred- Train Correct test Error- Early_IA_Correct ensemble': "S4D",
 'supp_CCG single class pred- Train Correct test Error- Early_RS_Error ensemble': "S4E",
 'supp_CCG single class pred- Train IA test RS - Early_RS_Correct ensemble': "S4C",
 # current '+null' fig_names emitted by SVM v5 uniproportion-CCG supplement cells (76-78)
 'supp_CCG+null single class pred- Train IA test RS - Early_RS_Correct ensemble': "S4C",
 'supp_CCG + null 1_class pred-Train_Correct_Test_Error- Early_RS_Error ensemble': "S4E",
 'supp_CCG + null 1_class pred-Train_Correct_Test_Error-Early_IA_Correct ensemble': "S4D"
                         }

legacy_map = {
    'null_CCGP- Train on IA trials, Test on RS trials_Early stage ens_ ': '3B',
    'null CCGP (early ens) train_correct_test_error': '4B',
    'null_CCG uni-class pred- Train IA test RS - Early_RS_Correct ensemble': '3C',
    'null_ CCG uni-class pred- Train Correct test Error- Early_IA_Correct ensemble': '4C',
    'null_CCG uni-class pred- Train Correct test Error- Early_RS_Error ensemble': '4F',
    'Supplement- VEH v CLNZ for Het & WT- ensemble proportion overlap': 'S1F',  
}


full_filename_panel_map = {**map_panel_to_filename, **map_supp_panel_to_file, **legacy_map}
full_filename_panel_map

{'Autoencoder Early_IA_Correct_v_Early_RS_Correct DB index by ensembles': '6G',
 'Autoencoder Early_IA_Correct_v_Late_IA DB index by ensembles': '7D',
 'Autoencoder Early_IA_Error_v_Early_RS_Error DB index by ensembles': '5H',
 'Autoencoder Late_IA_v_Early_RS_Correct DB index by ensembles': '7H',
 'CCG single class pred- Train Correct test Error- Early_IA_Correct ensemble': '4C',
 'CCG single class pred- Train Correct test Error- Early_RS_Error ensemble': '4F',
 'CCG single class pred- Train IA test RS - Early_RS_Correct ensemble': '3C',
 'Early_IA_Correct_v_Early_RS_Correct SVM accuracy by ensem': '6C',
 'Early_IA_Correct_v_Late_IA SVM accuracy by ensem': '7C',
 'Early_IA_Error_v_Early_RS_Error SVM accuracy by ensem': '5C',
 'Late_IA_v_Early_RS_Correct SVM accuracy by ensem': '7G',
 'time-dep decoding - Early RS Correct ens- Early_IA_Correct_v_Early_RS_Correct': '6D',
 'time-dep decoding - Early RS Error ens- Early_IA_Error_v_Early_RS_Error': '5D',
 'Mean % of cells active in stage th

#### prepare and save concat figure DF

In [59]:
def clean_pvalue_string(x:float): 
    if x < 0.0001:
        if x == 0:
            cleaned =    r"<< 1 x 10^15"
        else:
            cleaned =    f'{x:.2e}'.replace("e", " x 10^")
    else:
        cleaned = f"{x:.4f}"
    return cleaned
import re
panel_re = re.compile(r'^(s_)?(\d+)_?([A-Z]+)_')
## v2 update: to handle combo panels like "5F-H" where multiple letters are present, and to add "S" prefix for supp figs

def extract_panel_id(name): 
    if pd.isna(name):
        return None
    m = panel_re.match(name)
    if not m:
        return None
    supp_prefix = "S" if m.group(1) else ""
    fig_num = m.group(2)
    letters = m.group(3)
    # single letter → "5C"; combo → "5F-H"
    panel = letters if len(letters) == 1 else f"{letters[0]}-{letters[-1]}"
    return f"{supp_prefix}{fig_num}{panel}"


In [60]:
# fresh apply
full_fig_table["Figure Panel"] = full_fig_table['filename of source table'].apply(extract_panel_id)

# definitive view: each unique filename → what panel did it get
view = (full_fig_table.groupby('filename of source table')
        .agg(n_rows=('Figure Panel', 'size'),
             figure_panel=('Figure Panel', 'first')))
print(view)
print(f"\nTotal rows: {len(full_fig_table)}")
print(f"Rows with panel: {full_fig_table['Figure Panel'].notna().sum()}")


                                                    n_rows figure_panel
filename of source table                                               
5C_SVM classifier accuracy Early_IA_Error_v_Ear...       6           5C
5D_Early RS Error ensemble time-based classific...      18           5D
6C_SVM classifier accuracy Early_IA_Correct_v_E...       6           6C
6D_Early RS Correct ensemble time-based classif...      18           6D
7_C_SVM classifier accuracy Early_IA_Correct_v_...       6           7C
7_G_SVM classifier accuracy Late_IA_v_Early_RS_...       6           7G
CCG single class pred- Train IA test RS - whole...       6         None
Mean % of cells active in stage that are in sta...      24         None
null CCGP (early ens) train_correct_test_error          12         None
null_ CCG uni-class pred- Train Correct test Er...       6         None
null_CCG uni-class pred- Train Correct test Err...       6         None
null_CCG uni-class pred- Train IA test RS - Ear...       6      

In [61]:
# valid_panels = list(full_filename_panel_map.values())
valid_panels = sorted(set(map_panel_to_filename.values())
                     | set(map_supp_panel_to_file.values())
                     | set(legacy_map.values()))

print(f" Keeping rows with panel IDs: {valid_panels}")
# step 1: regex extraction for panel-prefixed filenames (5C_, 7_C_, s_4_C_, etc.)
full_fig_table["Figure Panel"] = full_fig_table['filename of source table'].apply(extract_panel_id)
# step 2: legacy-map fallback for filenames without panel prefix (null_*, supp_*, etc.)
full_fig_table["Figure Panel"] = full_fig_table["Figure Panel"].fillna(
    full_fig_table['filename of source table'].map(full_filename_panel_map)
)
# step 3: keep only rows with a mapped panel (preserves combo tags like 5F-H, 6E-G)
full_fig_table = full_fig_table[full_fig_table["Figure Panel"].notna()]
print(full_fig_table["Figure Panel"].value_counts())
full_fig_table
 
 
# # valid_panels = list(full_filename_panel_map.values())
# valid_panels = sorted(set(map_panel_to_filename.values())
#                      | set(map_supp_panel_to_file.values())
#                      | set(legacy_map.values()))

# print(f" Keeping rows with panel IDs: {valid_panels}")
# full_fig_table["Figure Panel"]= full_fig_table['filename of source table'].replace(full_filename_panel_map)
# full_fig_table = full_fig_table[full_fig_table["Figure Panel"].isin(valid_panels)]## make sure you only keep table with real panels of interest
# full_fig_table


 Keeping rows with panel IDs: ['3B', '3C', '4B', '4C', '4F', '5C', '5D', '5H', '6C', '6D', '6G', '7C', '7D', '7G', '7H', 'S1B', 'S1C', 'S1F', 'S1I', 'S4C', 'S4D', 'S4E', 'S5G', 'S5I']
Figure Panel
S1I    24
S1C    24
S1B    24
6D     18
5D     18
3B     12
4B     12
S4C     8
S4D     8
S4E     8
3C      6
4C      6
4F      6
5C      6
6C      6
7C      6
7G      6
Name: count, dtype: int64


,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,filename of source table,Figure number,csv_date,csv_filename,group_1_null_low,group_1_null_high,group_2_null_low,group_2_null_high,exceeds_null,Figure Panel
0,Early_IA_Correct,WT VEH,WT CLNZ,29,21,58.7353,2.9926,61.6694,3.3390,MWU,...,Mean % of cells active in stage that are in st...,1_s,25 May 2026,1_s_Mean % of cells active in stage that are i...,NaN,NaN,NaN,NaN,NaN,S1I
1,Early_IA_Correct,WT VEH,Het VEH,29,27,58.7353,2.9926,69.5951,2.8289,MWU,...,Mean % of cells active in stage that are in st...,1_s,25 May 2026,1_s_Mean % of cells active in stage that are i...,NaN,NaN,NaN,NaN,NaN,S1I
2,Early_IA_Correct,Het VEH,Het CLNZ,27,21,69.5951,2.8289,69.4418,2.9976,MWU,...,Mean % of cells active in stage that are in st...,1_s,25 May 2026,1_s_Mean % of cells active in stage that are i...,NaN,NaN,NaN,NaN,NaN,S1I
3,Early_IA_Correct,Het VEH,Het postCLNZ,27,26,69.5951,2.8289,70.1219,2.1584,MWU,...,Mean % of cells active in stage that are in st...,1_s,25 May 2026,1_s_Mean % of cells active in stage that are i...,NaN,NaN,NaN,NaN,NaN,S1I
4,Early_IA_Error,WT VEH,WT CLNZ,11,9,76.8790,6.4214,72.8287,6.0399,MWU,...,Mean % of cells active in stage that are in st...,1_s,25 May 2026,1_s_Mean % of cells active in stage that are i...,NaN,NaN,NaN,NaN,NaN,S1I
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,Early_RS_Correct,Het VEH,Het CLNZ,10000,10000,0.9532,0.0062,0.9669,0.0053,robust_cohen_d,...,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,7,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN,7G
2,Early_RS_Correct,Het VEH,Het postCLNZ,10000,10000,0.9532,0.0062,0.9769,0.0044,robust_cohen_d,...,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,7,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN,7G
3,Late_IA,WT VEH,Het VEH,10000,10000,0.9696,0.0051,0.9646,0.0053,robust_cohen_d,...,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,7,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN,7G
4,Late_IA,Het VEH,Het CLNZ,10000,10000,0.9646,0.0053,0.9668,0.0052,robust_cohen_d,...,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,7,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN,7G


In [62]:

## TEST RELATED PREPROCESS
#Rename varaible names
variable_map = {'prop_active_frames': "% of frames with events", 
                 'mean_rate': "Normalized event rate",
                 'active_in_trial': '% of cells active per trial',
                 'value': '% of samples classified',
                'accuracy':'SVM Accuracy',
                'mean_acc': 'Time-dependent SVM Accuracy',
                'DB_index':'Davies-Bouldin Index of Latent Space activity',
               }
full_fig_table['Variable Compared between Groups'] = full_fig_table['Variable Compared between Groups'].map(variable_map)
#find cohen's d 
cohen_d_mask =  full_fig_table['Name of Statistical Test'] == 'robust_cohen_d'
full_fig_table.loc[cohen_d_mask, 'Test Result'] = full_fig_table.loc[cohen_d_mask, 'Test Result'].str.replace(" ", ",").apply(ast.literal_eval) #aplpy to cohen's d testsing
#clean/redo testing 
test_name_clean = {'robust_cohen_d': "Robust Cohen's d",
                   "permutation_test": "Permutation Test",
                   'MWU': "Mann-Whitney U",
                   'chi_squared':"Chi-Squared"}
full_fig_table['Name of Statistical Test'] = full_fig_table['Name of Statistical Test'].map(test_name_clean) #replace var name with rea lnames 
#for MWU/permutation tests, replace first space occurance with comma then literal eval
test_stat_spaceless_mask = (full_fig_table['Name of Statistical Test'] == 'Permutation Test') | (full_fig_table['Name of Statistical Test'] == 'Mann-Whitney U')
full_fig_table.loc[test_stat_spaceless_mask, 'Test Result']= full_fig_table.loc[test_stat_spaceless_mask, 'Test Result'].str.replace(" ", ",", n = 1).apply(ast.literal_eval)
#clean p-values

full_fig_table['Test Statistic Variable'] = full_fig_table['Name of Statistical Test'].map({"Robust Cohen's d": "Robust Cohen's d",
                                                                                            "Permutation Test": "Mean permutation group diff.",
                                                                                            "Mann-Whitney U Test": "U-statistic",
                                                                                            "Chi-Squared Test": "Chi-Squared"})
full_fig_table['Test Statistic Value'] = full_fig_table['Test Result'].apply(lambda x: x[0])
full_fig_table['Test p-value'] = full_fig_table['Test p-value'].apply(lambda x: clean_pvalue_string(x))
#bugfix- force timebin in comparison string to avoid excel results as dates 
time_dep_rows = full_fig_table['filename of source table'].str.contains("time-dep")
full_fig_table.loc[time_dep_rows,'Group of posthoc comparison']= "timebins: "+ full_fig_table.loc[time_dep_rows,'Group of posthoc comparison']
full_fig_table.loc[time_dep_rows,'Alternate name- group compared within']= "timebins: "+ full_fig_table.loc[time_dep_rows,'Alternate name- group compared within']
full_fig_table.tail()


C:\Users\13car\AppData\Local\Temp\ipykernel_15524\3268371116.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  full_fig_table['Variable Compared between Groups'] = full_fig_table['Variable Compared between Groups'].map(variable_map)
C:\Users\13car\AppData\Local\Temp\ipykernel_15524\3268371116.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  full_fig_table['Name of Statistical Test'] = full_fig_table['Name of Statistical Test'].map(test_name_clean) #replace var name with rea lnames
C:\Users\13car\AppD

,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,csv_date,csv_filename,group_1_null_low,group_1_null_high,group_2_null_low,group_2_null_high,exceeds_null,Figure Panel,Test Statistic Variable,Test Statistic Value
1,Early_RS_Correct,Het VEH,Het CLNZ,10000,10000,0.9532,0.0062,0.9669,0.0053,Robust Cohen's d,...,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN,7G,Robust Cohen's d,2.37898
2,Early_RS_Correct,Het VEH,Het postCLNZ,10000,10000,0.9532,0.0062,0.9769,0.0044,Robust Cohen's d,...,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN,7G,Robust Cohen's d,4.41560
3,Late_IA,WT VEH,Het VEH,10000,10000,0.9696,0.0051,0.9646,0.0053,Robust Cohen's d,...,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN,7G,Robust Cohen's d,0.96613
4,Late_IA,Het VEH,Het CLNZ,10000,10000,0.9646,0.0053,0.9668,0.0052,Robust Cohen's d,...,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN,7G,Robust Cohen's d,0.41547
5,Late_IA,Het VEH,Het postCLNZ,10000,10000,0.9646,0.0053,0.9697,0.0052,Robust Cohen's d,...,04 Jun 2026,7_7_G_SVM classifier accuracy Late_IA_v_Early_...,NaN,NaN,NaN,NaN,NaN,7G,Robust Cohen's d,0.96593


In [63]:
## update with supplementary table 2 final format 
#reorder then drop figs
new_col_order = ['Figure Panel', 'Group of posthoc comparison', 'Group 1', 'Group 2','Variable Compared between Groups',
                 'Group 1 N', 'Group 2 N', 'Group 1 Mean', 'Group 1 SEM', 'Group 2 Mean', 'Group 2 SEM',
                 'Name of Statistical Test', 'Test p-value','Test Statistic Variable','Test Statistic Value',
                 'Variable labeling post-hoc group',
                 'Categorical variable of plot (x-axis)', 
                 'filename of source table',
                 ]

## final supplement table map
col_rename_map = {'Variable Compared between Groups': 'Dependent Variable',
                  'Name of Statistical Test': 'Statistical Test',
                  'Group of posthoc comparison': 'Posthoc Comparison Group',
                  'Test p-value': 'p-value'}
## carry through null band (5th/95th pct) raw columns when present (CCGP figs 3B/4B)
null_band_raw_cols = ['group_1_null_low', 'group_1_null_high', 'group_2_null_low', 'group_2_null_high']
new_col_order = new_col_order + [c for c in null_band_raw_cols if c in full_fig_table.columns]
full_fig_table = full_fig_table.loc[:, new_col_order].rename(columns = col_rename_map)
## combine 'Group 1' / 'Group 2' name columns into a single 'Group 1 & 2' column (joined with ' vs. '), in place
full_fig_table.insert(full_fig_table.columns.get_loc('Group 1'), 'Group 1 & 2',
                      full_fig_table.apply(lambda x: f"{x['Group 1']}-{x['Group 2']}", axis=1))
full_fig_table = full_fig_table.drop(columns=['Group 1', 'Group 2'])
full_fig_table['shorthand test name'] = full_fig_table['Statistical Test'].map({"Robust Cohen's d": "d","Mann-Whitney U": "U"})
full_fig_table['Test stat., Name, Value']= full_fig_table.apply(lambda x: f'{x['shorthand test name']}={round(x["Test Statistic Value"],3)}', axis=1)
# full_fig_table['Grouping Variable ']= full_fig_table['Variable Compared between Groups']

full_fig_table

,Figure Panel,Posthoc Comparison Group,Group 1 & 2,Dependent Variable,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,...,Test Statistic Value,Variable labeling post-hoc group,Categorical variable of plot (x-axis),filename of source table,group_1_null_low,group_1_null_high,group_2_null_low,group_2_null_high,shorthand test name,"Test stat., Name, Value"
0,S1I,Early_IA_Correct,WT VEH-WT CLNZ,NaN,29,21,58.7353,2.9926,61.6694,3.3390,...,285.50000,geno_day,task_phase_vec,Mean % of cells active in stage that are in st...,NaN,NaN,NaN,NaN,U,U=285.5
1,S1I,Early_IA_Correct,WT VEH-Het VEH,NaN,29,27,58.7353,2.9926,69.5951,2.8289,...,251.50000,geno_day,task_phase_vec,Mean % of cells active in stage that are in st...,NaN,NaN,NaN,NaN,U,U=251.5
2,S1I,Early_IA_Correct,Het VEH-Het CLNZ,NaN,27,21,69.5951,2.8289,69.4418,2.9976,...,288.00000,geno_day,task_phase_vec,Mean % of cells active in stage that are in st...,NaN,NaN,NaN,NaN,U,U=288.0
3,S1I,Early_IA_Correct,Het VEH-Het postCLNZ,NaN,27,26,69.5951,2.8289,70.1219,2.1584,...,356.50000,geno_day,task_phase_vec,Mean % of cells active in stage that are in st...,NaN,NaN,NaN,NaN,U,U=356.5
4,S1I,Early_IA_Error,WT VEH-WT CLNZ,NaN,11,9,76.8790,6.4214,72.8287,6.0399,...,55.00000,geno_day,task_phase_vec,Mean % of cells active in stage that are in st...,NaN,NaN,NaN,NaN,U,U=55.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,7G,Early_RS_Correct,Het VEH-Het CLNZ,SVM Accuracy,10000,10000,0.9532,0.0062,0.9669,0.0053,...,2.37898,geno_day,enriched_group,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,NaN,NaN,NaN,NaN,d,d=2.379
2,7G,Early_RS_Correct,Het VEH-Het postCLNZ,SVM Accuracy,10000,10000,0.9532,0.0062,0.9769,0.0044,...,4.41560,geno_day,enriched_group,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,NaN,NaN,NaN,NaN,d,d=4.416
3,7G,Late_IA,WT VEH-Het VEH,SVM Accuracy,10000,10000,0.9696,0.0051,0.9646,0.0053,...,0.96613,geno_day,enriched_group,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,NaN,NaN,NaN,NaN,d,d=0.966
4,7G,Late_IA,Het VEH-Het CLNZ,SVM Accuracy,10000,10000,0.9646,0.0053,0.9668,0.0052,...,0.41547,geno_day,enriched_group,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,NaN,NaN,NaN,NaN,d,d=0.415


In [64]:
## cleaning up posthoc comparisons
full_fig_table['Posthoc Comparison Group'] = full_fig_table['Posthoc Comparison Group'].str.replace("_", " ")
## time-dependent decoding clean up
time_dep_rows = full_fig_table['Posthoc Comparison Group'].str.contains("-")
full_fig_table.loc[time_dep_rows,'Posthoc Comparison Group'] = full_fig_table.loc[time_dep_rows,'Posthoc Comparison Group']+ " seconds from outcome"


In [65]:
## v2 update- combine columns for space
#  create group 1, 2 N column
full_fig_table['N- Group 1 & 2 '] = full_fig_table.apply(lambda x: f"{x['Group 1 N']}, {x['Group 2 N']}", axis=1)
#  create +/- sem col
full_fig_table['Group 1 Mean +/- SEM'] = (full_fig_table['Group 1 Mean'].map('{:.3f}'.format)
    + ' +/- '  + full_fig_table['Group 1 SEM'].map('{:.3f}'.format))
full_fig_table['Group 2 Mean +/- SEM'] = (full_fig_table['Group 2 Mean'].map('{:.3f}'.format)
    + ' +/- ' + full_fig_table['Group 2 SEM'].map('{:.3f}'.format))
## null band (5th-95th percentile) combined into one string per group, like Mean +/- SEM
null_band_pairs = {'Group 1 Null Band (5-95%)': ('group_1_null_low', 'group_1_null_high'),
                   'Group 2 Null Band (5-95%)': ('group_2_null_low', 'group_2_null_high')}
for band_col, (lo_col, hi_col) in null_band_pairs.items():
    if lo_col in full_fig_table.columns and hi_col in full_fig_table.columns:
        full_fig_table[band_col] = full_fig_table.apply(
            lambda x, lo=lo_col, hi=hi_col: f"{x[lo]:.3f} - {x[hi]:.3f}" if pd.notna(x[lo]) and pd.notna(x[hi]) else "",
            axis=1)
full_fig_table.head()

,Figure Panel,Posthoc Comparison Group,Group 1 & 2,Dependent Variable,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,...,group_1_null_high,group_2_null_low,group_2_null_high,shorthand test name,"Test stat., Name, Value",N- Group 1 & 2,Group 1 Mean +/- SEM,Group 2 Mean +/- SEM,Group 1 Null Band (5-95%),Group 2 Null Band (5-95%)
0,S1I,Early IA Correct,WT VEH-WT CLNZ,NaN,29,21,58.7353,2.9926,61.6694,3.3390,...,NaN,NaN,NaN,U,U=285.5,"29, 21",58.735 +/- 2.993,61.669 +/- 3.339,,
1,S1I,Early IA Correct,WT VEH-Het VEH,NaN,29,27,58.7353,2.9926,69.5951,2.8289,...,NaN,NaN,NaN,U,U=251.5,"29, 27",58.735 +/- 2.993,69.595 +/- 2.829,,
2,S1I,Early IA Correct,Het VEH-Het CLNZ,NaN,27,21,69.5951,2.8289,69.4418,2.9976,...,NaN,NaN,NaN,U,U=288.0,"27, 21",69.595 +/- 2.829,69.442 +/- 2.998,,
3,S1I,Early IA Correct,Het VEH-Het postCLNZ,NaN,27,26,69.5951,2.8289,70.1219,2.1584,...,NaN,NaN,NaN,U,U=356.5,"27, 26",69.595 +/- 2.829,70.122 +/- 2.158,,
4,S1I,Early IA Error,WT VEH-WT CLNZ,NaN,11,9,76.8790,6.4214,72.8287,6.0399,...,NaN,NaN,NaN,U,U=55.0,"11, 9",76.879 +/- 6.421,72.829 +/- 6.040,,


#### Save full figure table after concats 

In [66]:
#save fig table
cols_drop_in_save = ['Test Result', 
                     'Alternate name- group compared within', 
                     'comparison',
                     'Figure number',
                     'Date of figure creation',
                     'Variable labeling post-hoc group',
                     'Categorical variable of plot (x-axis)',
                     'shorthand test name', 
                     'Test Statistic Value',
                     'Test Statistic Variable',
                     'Group 1 N',
                    'Group 2 N',
                     'Group 1 Mean', 
                     'Group 1 SEM', 
                     'Group 2 Mean',
                    'Group 2 SEM',
                     'group_1_null_low', 'group_1_null_high',
                     'group_2_null_low', 'group_2_null_high'] ## drop any columns in this list that are still present in the table before saving, to avoid saving extraneous info

full_fig_table = full_fig_table.drop([c for c in cols_drop_in_save if c in full_fig_table.columns],axis = 1)
csv_name = f"all_figure_concat_results.csv"
full_fig_table.set_index("Figure Panel").to_csv( csv_store_folder/csv_name)
full_fig_table

,Figure Panel,Posthoc Comparison Group,Group 1 & 2,Dependent Variable,Statistical Test,p-value,filename of source table,"Test stat., Name, Value",N- Group 1 & 2,Group 1 Mean +/- SEM,Group 2 Mean +/- SEM,Group 1 Null Band (5-95%),Group 2 Null Band (5-95%)
0,S1I,Early IA Correct,WT VEH-WT CLNZ,NaN,Mann-Whitney U,0.7160,Mean % of cells active in stage that are in st...,U=285.5,"29, 21",58.735 +/- 2.993,61.669 +/- 3.339,,
1,S1I,Early IA Correct,WT VEH-Het VEH,NaN,Mann-Whitney U,0.0221,Mean % of cells active in stage that are in st...,U=251.5,"29, 27",58.735 +/- 2.993,69.595 +/- 2.829,,
2,S1I,Early IA Correct,Het VEH-Het CLNZ,NaN,Mann-Whitney U,0.9337,Mean % of cells active in stage that are in st...,U=288.0,"27, 21",69.595 +/- 2.829,69.442 +/- 2.998,,
3,S1I,Early IA Correct,Het VEH-Het postCLNZ,NaN,Mann-Whitney U,0.9291,Mean % of cells active in stage that are in st...,U=356.5,"27, 26",69.595 +/- 2.829,70.122 +/- 2.158,,
4,S1I,Early IA Error,WT VEH-WT CLNZ,NaN,Mann-Whitney U,0.7017,Mean % of cells active in stage that are in st...,U=55.0,"11, 9",76.879 +/- 6.421,72.829 +/- 6.040,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,7G,Early RS Correct,Het VEH-Het CLNZ,SVM Accuracy,Robust Cohen's d,0.0087,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,d=2.379,"10000, 10000",0.953 +/- 0.006,0.967 +/- 0.005,,
2,7G,Early RS Correct,Het VEH-Het postCLNZ,SVM Accuracy,Robust Cohen's d,5.04 x 10^-06,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,d=4.416,"10000, 10000",0.953 +/- 0.006,0.977 +/- 0.004,,
3,7G,Late IA,WT VEH-Het VEH,SVM Accuracy,Robust Cohen's d,0.1670,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,d=0.966,"10000, 10000",0.970 +/- 0.005,0.965 +/- 0.005,,
4,7G,Late IA,Het VEH-Het CLNZ,SVM Accuracy,Robust Cohen's d,0.3389,7_G_SVM classifier accuracy Late_IA_v_Early_RS...,d=0.415,"10000, 10000",0.965 +/- 0.005,0.967 +/- 0.005,,


In [67]:
full_fig_table['Figure Panel'].value_counts()

Figure Panel
S1I    24
S1C    24
S1B    24
6D     18
5D     18
3B     12
4B     12
S4C     8
S4D     8
S4E     8
3C      6
4C      6
4F      6
5C      6
6C      6
7C      6
7G      6
Name: count, dtype: int64

In [68]:
full_fig_table.iloc[:, -2:].value_counts()

Group 1 Null Band (5-95%)  Group 2 Null Band (5-95%)
                                                        174
0.411 - 0.588              0.431 - 0.569                  1
0.462 - 0.537              0.441 - 0.557                  1
                           0.430 - 0.568                  1
0.452 - 0.549              0.442 - 0.557                  1
                           0.439 - 0.561                  1
0.435 - 0.567              0.452 - 0.549                  1
0.435 - 0.566              0.431 - 0.570                  1
                           0.423 - 0.578                  1
0.422 - 0.582              0.462 - 0.537                  1
0.422 - 0.577              0.444 - 0.557                  1
                           0.433 - 0.568                  1
0.411 - 0.588              0.390 - 0.610                  1
0.322 - 0.677              0.393 - 0.608                  1
0.410 - 0.590              0.427 - 0.574                  1
                           0.400 - 0.596       